# SA Provincial Archives Digitisation Risk Analysis
## Which Archives Are Next?

---

> *On 24 March 2026, fire gutted the Botha Sigcau Building in Mthatha — a 13-storey
> landmark that housed 11 government departments and irreplaceable Eastern Cape records
> dating back to the pre-colonial era. It was not an isolated incident. Eleven prominent
> buildings in Mthatha have been gutted by fire since 2021. This analysis maps the
> digitisation risk across all nine South African provincial archive repositories
> and asks: which archives are next — and what must be done before another fire
> decides for us.*

---

## Source Register

All data points in this analysis are either directly sourced or estimated from a
confirmed national baseline. Every score in the CSV includes a `_source` column
documenting its origin. The following are the primary sources used:

| Source | URL | What it provides |
|--------|-----|------------------|
| NARSSA NAAIRS Introduction | national.archives.gov.za/naairsintro.htm | Confirms national digitisation <50% complete |
| NARSSA NAAIRS Migration | nationalarchives.gov.za/node/737 | Confirms 8.3M records in mid-migration |
| UCT Archive & Public Culture Research Initiative (2013) | humanities.uct.ac.za/apc | Lebowa archive condition documented |
| Wits People's Guide to Archives | wits.ac.za/history-workshop/archives-guide/ | All Bantustan archive locations |
| SA Society of Archivists | saarchivist.co.za | Botha Sigcau fire + records at risk |
| AtoM — Access to Memory | accesstomemory.org | Open-source archival platform used in SA |
| Wits Research Archives | researcharchives.wits.ac.za | AtoM adoption confirmed in SA |
| Open Secrets Collection, Wits HPRA | historicalpapers-atom.wits.ac.za/al3450 | Apartheid records not destroyed |
| PMG Parliamentary Records | pmg.org.za/committee-meeting/22093/ | Limpopo/NW underfunding + NAAIRS revamp |
| WC Govt NAAIRS page | d7.westerncape.gov.za/service/national-automated-archival-information-retrieval-system-naairs | WC archives digitisation confirmed |
| EWN / Daily Maverick / SowetanLive | Various | Fire incident timeline |

**Data transparency note:**
49% of data points are directly sourced. 51% are informed estimates anchored to
the confirmed national NAAIRS baseline of <50% digitisation completion.
Province-by-province digitisation progress is not publicly available at this granularity.
Recommended next step: PAIA requests to each provincial DSAC equivalent, or
direct enquiry to naairs@dac.gov.za for national-level breakdowns.

**Audience:** Data scientists, policy analysts, archivists, NARSSA, DSAC, SITA

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FFFFFF',
    'axes.edgecolor':   '#CCCCCC',
    'axes.labelcolor':  '#1e293b',
    'xtick.color':      '#334155',
    'ytick.color':      '#334155',
    'text.color':       '#1e293b',
    'grid.color':       '#e2e8f0',
    'grid.linestyle':   '--',
    'grid.alpha':       0.7,
    'font.family':      'sans-serif',
    'figure.dpi':       120,
})
GOLD='#f0a500'; TEAL='#00c9a7'; RED='#e05c5c'
ORANGE='#f97316'; GREEN='#34d399'; GREY='#6b7280'

np.random.seed(42)
print('Setup complete ✓')

---
## 1. Load & Validate Data

In [ ]:
df = pd.read_csv('data/provincial_archives.csv')

print(f'Dataset shape: {df.shape}')
print(f'Provinces: {df["province"].tolist()}')
print(f'\nRisk tier distribution:')
print(df['risk_tier'].value_counts())
print(f'\nScore range: {df["risk_score_10"].min()} – {df["risk_score_10"].max()}')

# Show source transparency
print('\n=== SOURCE TRANSPARENCY SUMMARY ===')
source_cols = [c for c in df.columns if c.endswith('_source')]
sourced   = sum(1 for col in source_cols for v in df[col] if str(v).startswith('Sourced'))
estimated = sum(1 for col in source_cols for v in df[col] if str(v).startswith('Estimated'))
print(f'Sourced data points:   {sourced}')
print(f'Estimated data points: {estimated}')
print(f'Sourced percentage:    {sourced/(sourced+estimated)*100:.0f}%')
print()
print('Columns with source documentation:')
for col in source_cols:
    print(f'  {col}')

df[['province','risk_score_10','risk_tier','digitisation_score',
    'physical_vulnerability_score','budget_score']].head(9)

---
## 2. Risk Score Overview
### Which provinces face the highest digitisation risk?

In [ ]:
df_sorted = df.sort_values('risk_score_10', ascending=True)

tier_colors = {
    'Critical Risk': RED,
    'Moderate Risk': ORANGE,
    'Lower Risk': GREEN,
}
bar_colors = [tier_colors[t] for t in df_sorted['risk_tier']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(df_sorted['province'], df_sorted['risk_score_10'],
               color=bar_colors, alpha=0.9)

for bar, val, tier in zip(bars, df_sorted['risk_score_10'], df_sorted['risk_tier']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val}/10 — {tier}', va='center', fontsize=9, color='white')

ax.axvline(x=6.5, color=RED, linestyle='--', lw=1.2, alpha=0.7,
           label='Critical Risk threshold (6.5)')
ax.axvline(x=3.5, color=ORANGE, linestyle='--', lw=1.2, alpha=0.7,
           label='Moderate Risk threshold (3.5)')

ax.set_xlabel('Composite Risk Score (0–10)', fontsize=11)
ax.set_title('SA Provincial Archives — Digitisation Risk Score\n'
             'Higher score = greater risk of irreversible archival loss',
             fontweight='bold', color='white', pad=12)
ax.set_xlim(0, 13)
ax.legend(fontsize=9)
ax.grid(axis='x', zorder=0)
plt.tight_layout()
plt.savefig('chart_risk_overview.png', dpi=150, bbox_inches='tight',
            facecolor='#FAFAFA')
plt.show()

print('KEY FINDING: 5 of 9 provinces are in Critical Risk territory.')
print('Eastern Cape scores 10/10 — the maximum possible risk score.')
print('Western Cape scores 0/10 — demonstrating the gap is closeable with investment.')

---
## 3. The Apartheid Map Finding
### Does the digital preservation gap follow the Bantustan geography?

In [ ]:
# Correlation: Bantustan burden vs composite risk score
corr = df['bantustan_burden'].corr(df['risk_score_10'])
print(f'Pearson correlation — Bantustan burden vs Risk score: {corr:.3f}')
print(f'R²: {corr**2:.3f}')
print()

fig, ax = plt.subplots(figsize=(10, 6))

scatter_colors = [tier_colors[t] for t in df['risk_tier']]
ax.scatter(df['bantustan_burden'], df['risk_score_10'],
           c=scatter_colors, s=180, alpha=0.9,
           edgecolors='white', linewidths=1, zorder=3)

# Regression line
m, b = np.polyfit(df['bantustan_burden'], df['risk_score_10'], 1)
x_line = np.linspace(0, 3.2, 100)
ax.plot(x_line, m * x_line + b, '--', color=GOLD, lw=2,
        label=f'OLS fit (r = {corr:.3f})', zorder=2)

# Annotate each province
for _, row in df.iterrows():
    ax.annotate(
        row['province'],
        (row['bantustan_burden'], row['risk_score_10']),
        textcoords='offset points', xytext=(8, 4),
        fontsize=8.5, color='white'
    )

ax.set_xlabel('Number of Former Bantustan Archives Absorbed', fontsize=11)
ax.set_ylabel('Composite Risk Score (0–10)', fontsize=11)
ax.set_title('The Apartheid Map Finding\n'
             'Provinces with more Bantustan archives face significantly higher digitisation risk',
             fontweight='bold', color='white', pad=12)
ax.legend(fontsize=10)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig('chart_apartheid_map.png', dpi=150, bbox_inches='tight',
            facecolor='#FAFAFA')
plt.show()

print(f'HEADLINE FINDING: r = {corr:.3f}, R² = {corr**2:.3f}')
print('The digital preservation gap in South Africa follows the apartheid map.')
print('Provinces bearing the most Bantustan archive burden have the least capacity')
print('to preserve them — and face the highest risk of irreversible loss.')

---
## 4. Dimension Analysis
### Breaking down what drives risk in each province

In [ ]:
# Radar/spider chart alternative — grouped bar chart of all 5 dimensions
dims = [
    ('physical_vulnerability_score', 'Physical\nVulnerability', RED),
    ('digitisation_score',           'Digitisation\nProgress',  TEAL),
    ('budget_score',                 'Budget\nAdequacy',        GOLD),
    ('staff_capacity_score',         'Staff\nCapacity',         GREEN),
    ('undigitised_volume_score',     'Undigitised\nVolume',     ORANGE),
]

df_sorted2 = df.sort_values('risk_score_10', ascending=False)
x = np.arange(len(df_sorted2))
width = 0.15

fig, ax = plt.subplots(figsize=(14, 6))

for i, (col, label, color) in enumerate(dims):
    offset = (i - 2) * width
    ax.bar(x + offset, df_sorted2[col], width,
           label=label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_sorted2['province'], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score (1–5)')
ax.set_ylim(0, 6)
ax.set_title('Risk Dimensions by Province — Sorted by Overall Risk Score\n'
             'Note: for Digitisation, Budget and Staff — lower score = worse outcome',
             fontweight='bold', color='white', pad=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart_dimensions.png', dpi=150, bbox_inches='tight',
            facecolor='#FAFAFA')
plt.show()

print('Eastern Cape scores maximum risk (5/5) on FOUR of five dimensions.')
print('Limpopo scores maximum risk (5/5) on undigitised volume — driven by')
print('three Bantustan archives (Lebowa, Gazankulu, Venda) in its repository.')

---
## 5. The Lebowa Case Study
### The most documented example of archival neglect in South Africa

In [ ]:
print('=== THE LEBOWA ARCHIVE — A DOCUMENTED CRISIS ===')
print()
print('Location:    Basement of old legislative buildings, Lebowakgomo')
print('             (former capital of the Lebowa Bantustan)')
print('Condition:   Spread across 4 rooms + passage walls')
print('             Only partially catalogued')
print('             Not digitised')
print('             At times described as a random heap of papers')
print('Province:    Limpopo (absorbed into Limpopo Provincial Archives)')
print('Period:      Records from 1970s Bantustan era to 1994')
print()
print('Source: Archive and Public Culture Research Initiative, UCT (2013)')
print()
print('=== LEBOWA DIGITISATION ACTION PLAN ===')
print()

action_plan = [
    ('Phase 1', '0–3 months',   'Physical stabilisation',
     'Install fire suppression + humidity monitoring in Lebowakgomo basement.\n'
     '    Pest control assessment. Prevent further physical deterioration\n'
     '    before any records are moved or digitised.\n'
     '    Responsible: DPWI + DSAC'),
    ('Phase 2', '3–6 months',   'Rapid cataloguing',
     'Deploy qualified archivists to create a basic inventory of what exists.\n'
     '    Box and label all loose records before digitisation begins.\n'
     '    Partner with Unisa and UL archivist programmes for capacity.\n'
     '    Responsible: NARSSA + Limpopo Provincial Archives'),
    ('Phase 3', '6–18 months',  'Digitisation',
     'Deploy high-volume scanners from the NARSSA 2020/21 stimulus equipment.\n'
     '    Recruit youth via Presidential Employment Stimulus — same model\n'
     '    that deployed 453 youth nationally in 2020–2022.\n'
     '    Responsible: NARSSA + DSAC + Presidency'),
    ('Phase 4', '18–24 months', 'Public access',
     'Upload digitised records to NAAIRS (National Automated Archival\n'
     '    Information Retrieval System) for public access.\n'
     '    Index by record type, date, department and subject.\n'
     '    Responsible: NARSSA'),
]

for phase, timeline, title, actions in action_plan:
    print(f'{phase} ({timeline}): {title}')
    print(f'    {actions}')
    print()

print('FUNDING PATHWAY:')
print('Presidential Employment Stimulus precedent — R30M allocated in 2020/21,')
print('deploying 453 youth. A targeted Lebowakgomo allocation via the same')
print('mechanism is the most immediately achievable route.')

---
## 6. Fire Incident Timeline
### The pattern of loss since 2020

In [ ]:
fire_incidents = pd.DataFrame([
    {'year': 2021, 'building': 'Multiple Mthatha govt buildings (1)',
     'province': 'Eastern Cape', 'records_at_risk': 'Provincial government records'},
    {'year': 2021, 'building': 'Multiple Mthatha govt buildings (2)',
     'province': 'Eastern Cape', 'records_at_risk': 'Provincial government records'},
    {'year': 2022, 'building': 'Parliament (Old Assembly)',
     'province': 'Western Cape',  'records_at_risk': 'National government records, offices'},
    {'year': 2022, 'building': 'UCT Libraries',
     'province': 'Western Cape',  'records_at_risk': 'Special collections, rare books, photographs'},
    {'year': 2025, 'building': 'Mthatha govt building',
     'province': 'Eastern Cape', 'records_at_risk': 'Provincial records'},
    {'year': 2026, 'building': 'Botha Sigcau Building',
     'province': 'Eastern Cape', 'records_at_risk': 'Pre-colonial EC records, Deeds, 11 dept files'},
])

fig, ax = plt.subplots(figsize=(12, 5))
year_counts = fire_incidents.groupby('year').size()

colors = [RED if y >= 2026 else ORANGE if y >= 2025 else GOLD
          for y in year_counts.index]
bars = ax.bar(year_counts.index.astype(str), year_counts.values,
              color=colors, alpha=0.9, width=0.5)

for bar, val in zip(bars, year_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha='center', fontsize=12, fontweight='bold', color='white')

ax.set_xlabel('Year')
ax.set_ylabel('Fire Incidents Affecting Govt Records')
ax.set_title('Fire Incidents at SA Government Buildings Containing Records (2021–2026)\n'
             'Red = 2026 (Botha Sigcau) · Orange = 2025 · Gold = earlier',
             fontweight='bold', color='white', pad=12)
ax.grid(axis='y', zorder=0)
ax.set_ylim(0, 4)
plt.tight_layout()
plt.savefig('chart_fire_timeline.png', dpi=150, bbox_inches='tight',
            facecolor='#FAFAFA')
plt.show()

print('6 fire incidents affecting government records since 2021.')
print('Eastern Cape accounts for 4 of 6 incidents.')
print('This is a pattern, not a coincidence.')

---
## 7. Key Findings & Policy Recommendations

In [ ]:
print('=' * 65)
print('KEY FINDINGS')
print('=' * 65)
print()
print('FINDING 1: 5 of 9 provinces are at Critical Risk')
print('  Eastern Cape (10.0), Limpopo (9.5), North West (7.0),')
print('  Mpumalanga (6.7), KwaZulu-Natal (6.6)')
print()
print('FINDING 2: The digital preservation gap follows the apartheid map')
print('  Bantustan burden correlates with risk score at r=0.835.')
print('  Provinces carrying former homeland archives have the least')
print('  capacity to preserve them.')
print()
print('FINDING 3: Nationally, more than half of all archival holdings')
print('  are NOT yet in NAAIRS (NARSSA, national.archives.gov.za/naairsintro.htm)')
print('  NARSSA is currently migrating 8.3M entries to a new database')
print('  (nationalarchives.gov.za/node/737) — system mid-migration.')
print()
print('FINDING 4: Eastern Cape faces a compounded crisis')
print('  Maximum scores on 4 of 5 dimensions. 3 fire incidents since 2021.')
print('  Transkei archives undigitised. Ciskei archives inaccessible')
print('  (private hands, 30+ years — Wits People\'s Guide to Archives).')
print()
print('FINDING 5: Limpopo carries the largest Bantustan archive burden')
print('  3 former Bantustan archives — Lebowa, Gazankulu, Venda.')
print('  Lebowa documented as undigitised heap in Lebowakgomo basement')
print('  (UCT Archive & Public Culture Research Initiative, 2013).')
print()
print('FINDING 6: The solution already exists and is free')
print('  AtoM (Access to Memory — accesstomemory.org) is open-source,')
print('  ICA-standards-based, and already deployed in South Africa')
print('  at Wits Historical Papers and the Western Cape Archives.')
print('  There is no technology barrier to adoption.')
print()
print('FINDING 7: Apartheid records were not destroyed')
print('  A persistent myth holds that apartheid state records were destroyed.')
print('  Open Secrets (Wits HPRA) confirms a vast collection remains in')
print('  public and private archives — much of it undigitised and at risk.')
print()
print('=' * 65)
print('POLICY RECOMMENDATIONS')
print('=' * 65)
print()
print('1. Declare Eastern Cape and Limpopo archives a national emergency')
print('   Activate NARSSA emergency protocols. Immediate physical assessment.')
print()
print('2. Adopt AtoM nationally as the standard digitisation platform')
print('   Free, open-source, ICA-compliant, already proven in SA.')
print('   Source: accesstomemory.org + researcharchives.wits.ac.za')
print()
print('3. Launch targeted Presidential Employment Stimulus for Lebowa')
print('   R30M precedent from 2020/21 — same model, Lebowakgomo specifically.')
print()
print('4. Complete the NAAIRS migration as a national priority')
print('   8.3M records mid-migration. >50% of holdings still uncatalogued.')
print('   Source: nationalarchives.gov.za/node/737')
print()
print('5. Submit PAIA requests for province-level digitisation data')
print('   51% of risk scores are estimated — province-level data must be')
print('   made public. Contact: naairs@dac.gov.za')
print()
print('6. Locate and recover the KwaNdebele archives')
print('   Parliamentary question to DSAC. NARSSA investigation mandate.')
print()
print('7. Investigate the Ciskei archives — private hands for 30+ years')
print('   PAIA request. Parliamentary accountability question.')
print()
print('8. Western Cape knowledge transfer programme')
print('   WC leads SA on AtoM and ECM digitisation. Formal transfer of')
print('   methodology to Critical Risk provinces.')

---

## Attribution & Authorship

**Analysis and code:** Lindiwe Songelwa

**Data sources:**
- Bantustan archive locations: Wits University People's Guide to Archives (Issuu)
- Lebowa archive condition: Archive and Public Culture Research Initiative, UCT (2013)
- Botha Sigcau fire: saarchivist.co.za + EWN + Daily Maverick (March 2026)
- NARSSA mandate: national.archives.gov.za
- Digitisation funding: DSAC Parliamentary Questions 2020/21
- Ciskei archive status: Wits People's Guide to Archives
- SA Constitution Schedule 5: constitution.org.za

**Inspired by:** The March 2026 fire at the Botha Sigcau Building, Mthatha —
a building that served as the administrative heart of the Transkei government,
the site of Bantu Holomisa's 1987 coup, and a cornerstone of South Africa's
political and administrative history.

**Project:** SA Provincial Archives Digitisation Risk Dashboard
**Repo:** github.com/Lindiwe-22/sa-archives-risk
**App:** Streamlit Cloud deployment